<a href="https://colab.research.google.com/github/Aviijoshi/IMDB-reviews-sentiment-analysis/blob/main/IMDB_reviews.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [26]:
!pip install kaggle

**Importing the dependencies**


In [27]:
import os
import json

from zipfile import  ZipFile
import pandas as pd
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense,Embedding,LSTM
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.layers import Input

**Data collection-Kaggle API**


In [28]:
kaggle_dictionary =json.load(open("/content/kaggle.json"))

In [29]:
#Setup kaggle credentials as env variables
os.environ["KAGGLE_USERNAME"] = kaggle_dictionary["username"]
os.environ["KAGGLE_KEY"] = kaggle_dictionary["key"]

In [30]:
!kaggle datasets download lakshmi25npathi/imdb-dataset-of-50k-movie-reviews

Dataset URL: https://www.kaggle.com/datasets/lakshmi25npathi/imdb-dataset-of-50k-movie-reviews
License(s): other
imdb-dataset-of-50k-movie-reviews.zip: Skipping, found more recently modified local copy (use --force to force download)


In [31]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


In [32]:
#Unzip the dataset file
with ZipFile("imdb-dataset-of-50k-movie-reviews.zip","r") as zip_ref:
  zip_ref.extractall()

In [33]:
!ls

'IMDB Dataset.csv'			 kaggle.json
 imdb-dataset-of-50k-movie-reviews.zip	 sample_data


**Loading the dataset**

In [34]:
data = pd.read_csv("/content/IMDB Dataset.csv")

In [35]:
data.shape

(50000, 2)

In [36]:
data.head()

,review,sentiment
0,One of the other reviewers has mentioned that ...,positive
1,A wonderful little production. <br /><br />The...,positive
2,I thought this was a wonderful way to spend ti...,positive
3,Basically there's a family where a little boy ...,negative
4,"Petter Mattei's ""Love in the Time of Money"" is...",positive


In [37]:
data["sentiment"].value_counts()

,count
sentiment,
positive,25000
negative,25000


In [38]:
data.replace({"sentiment":{"positive":1,"negative":0}},inplace=True)

/tmp/ipykernel_669/3877186056.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  data.replace({"sentiment":{"positive":1,"negative":0}},inplace=True)


In [39]:
data["sentiment"].value_counts()

,count
sentiment,
1,25000
0,25000


Spliting dataset into TEST AND TRAIN

In [40]:
train_data,test_data = train_test_split(data,test_size=0.2,random_state = 42)

In [41]:
print(train_data.shape)
print(test_data.shape)

(40000, 2)
(10000, 2)


**Data preprocessing**

In [42]:
#Tokenize text data
tokenizer = Tokenizer(num_words = 5000)
tokenizer.fit_on_texts(train_data["review"])
X_train = pad_sequences(tokenizer.texts_to_sequences(train_data["review"]),maxlen=200)
X_test = pad_sequences(tokenizer.texts_to_sequences(test_data["review"]),maxlen=200)

In [43]:
y_train = train_data["sentiment"]
y_test = test_data["sentiment"]

**BUILDING THE LSTM**


In [46]:
model = Sequential()

model.add(Input(shape=(200,)))
model.add(Embedding(input_dim=5000, output_dim=128))
model.add(LSTM(128, dropout=0.2, recurrent_dropout=0.2))
model.add(Dense(1, activation="sigmoid"))

model.summary()

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 200, 128)       │       640,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_3 (LSTM)                   │ (None, 128)            │       131,584 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 1)              │           129 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 771,713 (2.94 MB)

 Trainable params: 771,713 (2.94 MB)

 Non-trainable params: 0 (0.00 B)

In [47]:
#Compile the model
model.compile(optimizer="adam",loss="binary_crossentropy",metrics=["accuracy"]  )

**Training the model**

In [49]:
model.fit(X_train,y_train,epochs = 5,batch_size=64,validation_split=0.2)

Epoch 1/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 416s 825ms/step - accuracy: 0.7668 - loss: 0.4886 - val_accuracy: 0.8497 - val_loss: 0.3616
Epoch 2/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 386s 771ms/step - accuracy: 0.8547 - loss: 0.3473 - val_accuracy: 0.8562 - val_loss: 0.3419
Epoch 3/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 354s 707ms/step - accuracy: 0.8821 - loss: 0.2914 - val_accuracy: 0.8508 - val_loss: 0.3413
Epoch 4/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 356s 713ms/step - accuracy: 0.8940 - loss: 0.2655 - val_accuracy: 0.8709 - val_loss: 0.3162
Epoch 5/5
500/500 ━━━━━━━━━━━━━━━━━━━━ 393s 736ms/step - accuracy: 0.9054 - loss: 0.2391 - val_accuracy: 0.8752 - val_loss: 0.3234


**Model evaluation**

In [51]:
loss,accuracy = model.evaluate(X_test,y_test)
print(f"Test Loss:{loss}")
print(f"Test Accuracy:{accuracy}")

313/313 ━━━━━━━━━━━━━━━━━━━━ 31s 94ms/step - accuracy: 0.8777 - loss: 0.3127
Test Loss:0.3126966655254364
Test Accuracy:0.8776999711990356


**Building A predictive System**

In [69]:
def predict_sentiment(review):
  #Tokenized and pad the review
  sequence = tokenizer.texts_to_sequences([review])
  padded_sequence = pad_sequences(sequence,maxlen=200)
  prediction = model.predict(padded_sequence)
  print("Prediction probability:", prediction[0][0])
  sentiment = 'Positive' if prediction[0][0]> 0.5 else "Negative"
  return sentiment


In [72]:
#Example usage
new_review = "This movie was good"
sentiment = predict_sentiment(new_review)
print(f"The sentiment of the review is:{sentiment}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 70ms/step
Prediction probability: 0.46146867
The sentiment of the review is:Negative
